# ex
https://nflsavant.com/about.php

In [1]:
.load ../../dist/debug/jiff0
drop table if exists x;
create virtual table x using csv(
  filename="pbp-2024.csv",
  GameId,
  GameDate,
  Quarter integer,
  Minute integer,
  Second integer,
  OffenseTeam,
  DefenseTeam,
  Down,
  ToGo,
  YardLine,
  _1,
  SeriesFirstDown,
  _2,
  NextScore,
  Description,
  TeamWin,
  _3,
  _4,
  SeasonYear,
  Yards,
  Formation,
  PlayType,
  IsRush boolean,
  IsPass boolean,
  IsIncomplete boolean,
  IsTouchdown boolean,
  PassType,
  IsSack boolean,
  IsChallenge boolean,
  IsChallengeReversed boolean,
  Challenger,
  IsMeasurement boolean,
  IsInterception boolean,
  IsFumble boolean,
  IsPenalty boolean,
  IsTwoPointConversion boolean,
  IsTwoPointConversionSuccessful boolean,
  RushDirection,
  YardLineFixed,
  YardLineDirection,
  IsPenaltyAccepted boolean,
  PenaltyTeam,
  IsNoPlay boolean,
  PenaltyType,
  PenaltyYards
);

┌├

In [2]:
select typeof(IsTouchdown) from x limit 2;

typeof(IsTouchdown)
integer
integer


In [3]:
drop table if exists touchdowns;
create table touchdowns as 
select 
  GameId,
  GameDate,
  OffenseTeam,
  DefenseTeam, 
  Quarter,
  Minute,
  Second,
  iif(
    Quarter = 4 and Minute = 0, 
    jiff_time('01:00:00'),
    jiff_time(
      0, 
      (15 - Minute) + (Quarter-1) * 15,
      Second
    )
  )  as game_time,
  IsTouchdown
from x 
where Quarter != 5 and IsTouchdown;
select * from touchdowns limit 20;

GameId,GameDate,OffenseTeam,DefenseTeam,Quarter,Minute,Second,game_time,IsTouchdown
2024122907,2024-12-29,MIN,GB,3,9,50,00:36:50,1
2025010401,2025-01-04,PIT,CIN,2,9,57,00:21:57,1
2024122909,2024-12-29,TB,CAR,3,5,52,00:40:52,1
2024122906,2024-12-29,JAX,TEN,2,9,6,00:21:06,1
2024122903,2024-12-29,NYG,IND,4,3,5,00:57:05,1
2024122903,2024-12-29,IND,NYG,4,6,43,00:54:43,1
2024122903,2024-12-29,IND,NYG,4,10,57,00:50:57,1
2024122905,2024-12-29,BUF,NYJ,3,0,21,00:45:21,1
2024122905,2024-12-29,BUF,NYJ,3,1,17,00:44:17,1
2024122904,2024-12-29,ATL,WAS,4,1,23,00:59:23,1


In [4]:
select * from touchdowns order by game_time limit 10;

GameId,GameDate,OffenseTeam,DefenseTeam,Quarter,Minute,Second,game_time,IsTouchdown
2024102003,2024-10-20,CIN,CLE,1,15,0,00:00:00,1
2024091509,2024-09-15,WAS,NYG,1,15,0,00:00:00,1
2024111009,2024-11-10,ARI,NYJ,1,14,0,00:01:00,1
2024112402,2024-11-24,HOU,TEN,1,14,47,00:01:47,1
2024092200,2024-09-22,CLE,NYG,1,14,55,00:01:55,1
2024111800,2024-11-18,HOU,DAL,1,14,55,00:01:55,1
2024111009,2024-11-10,ARI,NYJ,1,13,1,00:02:01,1
2024111005,2024-11-10,NO,ATL,1,13,2,00:02:02,1
2024100607,2024-10-06,LV,DEN,1,13,8,00:02:08,1
2024100608,2024-10-06,ARI,SF,1,13,20,00:02:20,1


In [5]:
select 
  jiff_time_round(
    game_time, 
    'smallest', 'minute', 
    'mode', 'floor'
  ) as minute_bucket, 
  count(*)
from touchdowns
group by 1
order by 1;

minute_bucket,count(*)
00:00:00,2
00:01:00,4
00:02:00,7
00:03:00,21
00:04:00,20
00:05:00,22
00:06:00,22
00:07:00,20
00:08:00,19
00:09:00,22


In [9]:
select jiff_span_total(jiff_until('midnight', '00:05:00'), 's');

"jiff_span_total(jiff_until('midnight', '00:05:00'), 's')"
300


In [ ]:
select 
  jiff_time_round(time, 'minute'),
  count(*)
from touchdowns
group by 1
order by 1;

In [ ]:
select * from x where Quarter = '4' and Minute = '15' and IsTouchdown = '1' limit 10;